# Figure 5: Native RNA vs IVT RNA coverage correlation

Validates that IVT RNA faithfully reproduces native RNA coverage patterns across paired replicates. Set the data path in the cell below before running.

**Samples:** Native RNA (3 replicates), IVT RNA (3 replicates) — paired (Native 1 → IVT 1, Native 2 → IVT 2, Native 3 → IVT 3)

**Reference:** *E. coli* K-12 (ENA|U00096|U00096.3)

### Figures in this notebook: Native RNA vs IVT RNA Coverage Correlation

| Figure | Description |
|--------|-------------|
| **Fig. 5 (hexbin)** | Hexbin density plots — Native vs IVT per paired replicate |
| **Fig. 5 (scatter)** | Scatter plots — Native vs IVT per paired replicate |

All figures generated at three resolutions: per-base, 100 bp bins, 1 kb bins.

---

### Requirements

- **Python** 3.10+ with `matplotlib`, `pandas`, `numpy`, `scipy`
- **Read counts**: obtained with `samtools view -c -F 2308` (primary alignments only) — hardcoded in the read counts cell

### Input file layout

Place the notebook and the following files/folders in the same directory:

```
your_folder/
├── fig5_native_vs_ivt_correlation.ipynb
└── coverages/
    ├── native/
    │   ├── k12_native_full_bc1.txt
    │   ├── k12_native_full_bc2.txt
    │   └── k12_native_full_bc3.txt
    └── ivt/
        ├── k12_ivt_full_bc1.txt
        ├── k12_ivt_full_bc2.txt
        └── k12_ivt_full_bc3.txt
```

Coverage files are tab-separated with columns `chrom`, `position`, `depth` (standard `samtools depth` output). Output figures are saved to `per_base/`, `binned_100bp/`, and `binned_1kb/` subfolders.

> **Note:** `DATA_DIR` resolves to the directory Jupyter was launched from, which is normally the notebook's folder. If files fail to load, set `DATA_DIR` explicitly to an absolute path.

In [70]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib.gridspec import GridSpec
from pathlib import Path

plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['figure.dpi'] = 600

# set to the directory containing coverages/
DATA_DIR = Path('.').resolve()
# DATA_DIR = Path('/absolute/path/to/your/data')

## Load data

In [71]:
def load_coverage_file(filepath):
    df = pd.read_csv(filepath, sep='\t', header=None,
                     names=['chromosome', 'position', 'depth'])
    print(f"  {filepath.name}: {len(df):,} positions")
    return df

native1 = load_coverage_file(DATA_DIR / 'coverages/native/k12_native_full_bc1.txt')
native2 = load_coverage_file(DATA_DIR / 'coverages/native/k12_native_full_bc2.txt')
native3 = load_coverage_file(DATA_DIR / 'coverages/native/k12_native_full_bc3.txt')

ivt1 = load_coverage_file(DATA_DIR / 'coverages/ivt/k12_ivt_full_bc1.txt')
ivt2 = load_coverage_file(DATA_DIR / 'coverages/ivt/k12_ivt_full_bc2.txt')
ivt3 = load_coverage_file(DATA_DIR / 'coverages/ivt/k12_ivt_full_bc3.txt')

  k12_native_full_bc1.txt: 4,641,652 positions
  k12_native_full_bc2.txt: 4,641,652 positions
  k12_native_full_bc3.txt: 4,641,652 positions
  k12_ivt_full_bc1.txt: 4,641,652 positions
  k12_ivt_full_bc2.txt: 4,641,652 positions
  k12_ivt_full_bc3.txt: 4,641,652 positions


## CPM normalisation

In [72]:
# read counts from samtools view -c -F 2308
read_counts = {
    'native1': 88366,
    'native2': 235984,
    'native3': 347917,
    'ivt1': 72626,
    'ivt2': 50142,
    'ivt3': 49260
}

def calculate_cpm(df, total_reads):
    # CPM normalises by read count, but per-base coverage also scales with read length.
    # IVT reads (~428 bp) are shorter than native reads (~670 bp), so IVT will show
    # lower per-base CPM than native even after normalisation — this is expected.
    df_copy = df.copy()
    df_copy['cpm'] = (df_copy['depth'] / total_reads) * 1_000_000
    return df_copy

native1 = calculate_cpm(native1, read_counts['native1'])
native2 = calculate_cpm(native2, read_counts['native2'])
native3 = calculate_cpm(native3, read_counts['native3'])
ivt1 = calculate_cpm(ivt1, read_counts['ivt1'])
ivt2 = calculate_cpm(ivt2, read_counts['ivt2'])
ivt3 = calculate_cpm(ivt3, read_counts['ivt3'])

## Binning

In [73]:
def bin_coverage(df, bin_size):
    binned = df.copy()
    binned['bin_start'] = ((binned['position'] - 1) // bin_size) * bin_size + 1
    agg = binned.groupby('bin_start', sort=True).agg(
        cpm=('cpm', 'mean')
    ).reset_index()
    agg['position'] = agg['bin_start'] + (bin_size - 1) / 2.0
    return agg


def merge_binned_samples(df1, df2, bin_size, name1, name2):
    df1_binned = bin_coverage(df1, bin_size)
    df2_binned = bin_coverage(df2, bin_size)
    merged = pd.merge(
        df1_binned[['position', 'cpm']],
        df2_binned[['position', 'cpm']],
        on='position',
        suffixes=(f'_{name1}', f'_{name2}')
    )
    print(f"  {name1} vs {name2} ({bin_size}bp bins): {len(merged):,} bins")
    return merged

## Merge paired samples

In [74]:
def merge_two_samples(df1, df2, name1, name2):
    merged = pd.merge(
        df1[['position', 'cpm']],
        df2[['position', 'cpm']],
        on='position',
        suffixes=(f'_{name1}', f'_{name2}')
    )
    print(f"  {name1} vs {name2}: {len(merged):,} positions")
    return merged

# per-base
native1_ivt1 = merge_two_samples(native1, ivt1, 'Native Rep 1', 'IVT Rep 1')
native2_ivt2 = merge_two_samples(native2, ivt2, 'Native Rep 2', 'IVT Rep 2')
native3_ivt3 = merge_two_samples(native3, ivt3, 'Native Rep 3', 'IVT Rep 3')

# 100 bp bins
native1_ivt1_100bp = merge_binned_samples(native1, ivt1, 100, 'Native Rep 1', 'IVT Rep 1')
native2_ivt2_100bp = merge_binned_samples(native2, ivt2, 100, 'Native Rep 2', 'IVT Rep 2')
native3_ivt3_100bp = merge_binned_samples(native3, ivt3, 100, 'Native Rep 3', 'IVT Rep 3')

# 1 kb bins
native1_ivt1_1kb = merge_binned_samples(native1, ivt1, 1000, 'Native Rep 1', 'IVT Rep 1')
native2_ivt2_1kb = merge_binned_samples(native2, ivt2, 1000, 'Native Rep 2', 'IVT Rep 2')
native3_ivt3_1kb = merge_binned_samples(native3, ivt3, 1000, 'Native Rep 3', 'IVT Rep 3')

  Native Rep 1 vs IVT Rep 1: 4,641,652 positions
  Native Rep 2 vs IVT Rep 2: 4,641,652 positions
  Native Rep 3 vs IVT Rep 3: 4,641,652 positions
  Native Rep 1 vs IVT Rep 1 (100bp bins): 46,417 bins
  Native Rep 2 vs IVT Rep 2 (100bp bins): 46,417 bins
  Native Rep 3 vs IVT Rep 3 (100bp bins): 46,417 bins
  Native Rep 1 vs IVT Rep 1 (1000bp bins): 4,642 bins
  Native Rep 2 vs IVT Rep 2 (1000bp bins): 4,642 bins
  Native Rep 3 vs IVT Rep 3 (1000bp bins): 4,642 bins


## Correlations

In [75]:
def calculate_native_ivt_correlation(merged_df, name1, name2):
    cov1 = merged_df[f'cpm_{name1}'].values
    cov2 = merged_df[f'cpm_{name2}'].values

    # only positions with coverage in both samples
    mask = (cov1 > 0) & (cov2 > 0)
    cov1_filtered = cov1[mask]
    cov2_filtered = cov2[mask]

    spearman_r, spearman_p = stats.spearmanr(cov1_filtered, cov2_filtered)
    pearson_r, pearson_p = stats.pearsonr(cov1_filtered, cov2_filtered)

    n_total = len(cov1)
    n_both_nonzero = len(cov1_filtered)

    print(f"{name1} vs {name2}: Spearman ρ = {spearman_r:.3f}, Pearson r = {pearson_r:.3f} "
          f"({n_both_nonzero:,} positions, {100*n_both_nonzero/n_total:.1f}%)")

    return {
        'comparison': f'{name1} vs {name2}',
        'n_positions': n_both_nonzero,
        'n_total': n_total,
        'pct_covered_both': 100*n_both_nonzero/n_total,
        'spearman_r': spearman_r,
        'spearman_p': spearman_p,
        'pearson_r': pearson_r,
        'pearson_p': pearson_p
    }

print("Per-base:")
stats_n1_i1 = calculate_native_ivt_correlation(native1_ivt1, 'Native Rep 1', 'IVT Rep 1')
stats_n2_i2 = calculate_native_ivt_correlation(native2_ivt2, 'Native Rep 2', 'IVT Rep 2')
stats_n3_i3 = calculate_native_ivt_correlation(native3_ivt3, 'Native Rep 3', 'IVT Rep 3')

print("\n100bp bins:")
stats_n1_i1_100bp = calculate_native_ivt_correlation(native1_ivt1_100bp, 'Native Rep 1', 'IVT Rep 1')
stats_n2_i2_100bp = calculate_native_ivt_correlation(native2_ivt2_100bp, 'Native Rep 2', 'IVT Rep 2')
stats_n3_i3_100bp = calculate_native_ivt_correlation(native3_ivt3_100bp, 'Native Rep 3', 'IVT Rep 3')

print("\n1kb bins:")
stats_n1_i1_1kb = calculate_native_ivt_correlation(native1_ivt1_1kb, 'Native Rep 1', 'IVT Rep 1')
stats_n2_i2_1kb = calculate_native_ivt_correlation(native2_ivt2_1kb, 'Native Rep 2', 'IVT Rep 2')
stats_n3_i3_1kb = calculate_native_ivt_correlation(native3_ivt3_1kb, 'Native Rep 3', 'IVT Rep 3')

Per-base:
Native Rep 1 vs IVT Rep 1: Spearman ρ = 0.631, Pearson r = 0.878 (484,461 positions, 10.4%)
Native Rep 2 vs IVT Rep 2: Spearman ρ = 0.638, Pearson r = 0.884 (506,696 positions, 10.9%)
Native Rep 3 vs IVT Rep 3: Spearman ρ = 0.629, Pearson r = 0.892 (542,924 positions, 11.7%)

100bp bins:
Native Rep 1 vs IVT Rep 1: Spearman ρ = 0.590, Pearson r = 0.883 (6,115 positions, 13.2%)
Native Rep 2 vs IVT Rep 2: Spearman ρ = 0.560, Pearson r = 0.887 (6,385 positions, 13.8%)
Native Rep 3 vs IVT Rep 3: Spearman ρ = 0.534, Pearson r = 0.895 (6,842 positions, 14.7%)

1kb bins:
Native Rep 1 vs IVT Rep 1: Spearman ρ = 0.616, Pearson r = 0.927 (1,406 positions, 30.3%)
Native Rep 2 vs IVT Rep 2: Spearman ρ = 0.574, Pearson r = 0.922 (1,448 positions, 31.2%)
Native Rep 3 vs IVT Rep 3: Spearman ρ = 0.613, Pearson r = 0.930 (1,524 positions, 32.8%)


In [76]:
datasets = {
    'per_base': {
        'data': [
            (native1_ivt1, stats_n1_i1, 'Native Rep 1', 'IVT Rep 1'),
            (native2_ivt2, stats_n2_i2, 'Native Rep 2', 'IVT Rep 2'),
            (native3_ivt3, stats_n3_i3, 'Native Rep 3', 'IVT Rep 3')
        ],
        'suffix': '',
        'label': 'Per-base',
        'stats_list': [stats_n1_i1, stats_n2_i2, stats_n3_i3]
    },
    'binned_100bp': {
        'data': [
            (native1_ivt1_100bp, stats_n1_i1_100bp, 'Native Rep 1', 'IVT Rep 1'),
            (native2_ivt2_100bp, stats_n2_i2_100bp, 'Native Rep 2', 'IVT Rep 2'),
            (native3_ivt3_100bp, stats_n3_i3_100bp, 'Native Rep 3', 'IVT Rep 3')
        ],
        'suffix': '_100bp',
        'label': '100bp bins',
        'stats_list': [stats_n1_i1_100bp, stats_n2_i2_100bp, stats_n3_i3_100bp]
    },
    'binned_1kb': {
        'data': [
            (native1_ivt1_1kb, stats_n1_i1_1kb, 'Native Rep 1', 'IVT Rep 1'),
            (native2_ivt2_1kb, stats_n2_i2_1kb, 'Native Rep 2', 'IVT Rep 2'),
            (native3_ivt3_1kb, stats_n3_i3_1kb, 'Native Rep 3', 'IVT Rep 3')
        ],
        'suffix': '_1kb',
        'label': '1kb bins',
        'stats_list': [stats_n1_i1_1kb, stats_n2_i2_1kb, stats_n3_i3_1kb]
    }
}

## Plots

In [77]:
def plot_hexbin_panel(ax, merged_df, name1, name2, stats_dict, title, panel_label):
    cov1 = merged_df[f'cpm_{name1}'].values
    cov2 = merged_df[f'cpm_{name2}'].values

    mask = (cov1 > 0) & (cov2 > 0)
    cov1_log = np.log10(cov1[mask] + 1)
    cov2_log = np.log10(cov2[mask] + 1)

    hexbin = ax.hexbin(cov1_log, cov2_log, gridsize=50, cmap='Purples',
                       mincnt=1, bins='log')

    max_val = max(cov1_log.max(), cov2_log.max())
    ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, alpha=0.5)

    ax.set_xlabel(f'{name1.replace(" Rep ", " Rep")} coverage [log\u2081\u2080(CPM + 1)]', fontsize=17)
    ax.set_ylabel(f'{name2.replace(" Rep ", " Rep")} coverage [log\u2081\u2080(CPM + 1)]', fontsize=17)
    
    ax.set_title(title, fontsize=18, fontweight='bold')

    stats_text = (
        f"Positions: {stats_dict['n_positions']:,}\n"
        f"Spearman \u03c1: {stats_dict['spearman_r']:.3f}\n"
        f"Pearson r: {stats_dict['pearson_r']:.3f}"
    )
    ax.text(0.98, 0.02, stats_text, transform=ax.transAxes,
            fontsize=16, va='bottom', ha='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9))

    ax.text(-0.12, 1.05, panel_label, transform=ax.transAxes,
            fontsize=16, fontweight='bold', va='top', ha='right')

    ax.grid(True, alpha=0.3, linestyle='--')

    cbar = plt.colorbar(hexbin, ax=ax)
    cbar.set_label('Position count (log scale)', fontsize=18)


def plot_scatter_panel(ax, merged_df, name1, name2, stats_dict, title, panel_label):
    cov1 = merged_df[f'cpm_{name1}'].values
    cov2 = merged_df[f'cpm_{name2}'].values

    mask = (cov1 > 0) & (cov2 > 0)
    cov1_log = np.log10(cov1[mask] + 1)
    cov2_log = np.log10(cov2[mask] + 1)

    ax.scatter(cov1_log, cov2_log, s=0.5, alpha=0.3, c='#7B68EE',
               edgecolors='none', rasterized=True)

    max_val = max(cov1_log.max(), cov2_log.max())
    ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, alpha=0.7)

    ax.set_xlabel(f'{name1.replace(" Rep ", " Rep")} coverage [log\u2081\u2080(CPM + 1)]', fontsize=17)
    ax.set_ylabel(f'{name2.replace(" Rep ", " Rep")} coverage [log\u2081\u2080(CPM + 1)]', fontsize=17)
    ax.set_title(title, fontsize=18, fontweight='bold')

    stats_text = (
        f"Positions: {stats_dict['n_positions']:,}\n"
        f"Spearman \u03c1: {stats_dict['spearman_r']:.3f}\n"
        f"Pearson r: {stats_dict['pearson_r']:.3f}"
    )
    ax.text(0.98, 0.02, stats_text, transform=ax.transAxes,
            fontsize=16, va='bottom', ha='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9))

    ax.text(-0.12, 1.05, panel_label, transform=ax.transAxes,
            fontsize=16, fontweight='bold', va='top', ha='right')

    ax.grid(True, alpha=0.3, linestyle='--')

In [78]:
panel_labels = ['a', 'b', 'c']

for resolution, config in datasets.items():
    out_dir = Path(resolution)
    out_dir.mkdir(exist_ok=True)

    suffix = config['suffix']
    label = config['label']
    data_list = config['data']

    # hexbin
    fig = plt.figure(figsize=(18, 6))
    gs = GridSpec(1, 3, figure=fig, hspace=0.3, wspace=0.40)
    for idx, (merged_df, stats_dict, name1, name2) in enumerate(data_list):
        ax = fig.add_subplot(gs[0, idx])
        title = f'{name1.replace(" Rep ", " Rep")} vs {name2.replace(" Rep ", " Rep")}'
        plot_hexbin_panel(ax, merged_df, name1, name2, stats_dict, title, panel_labels[idx])
    fig.suptitle('', fontsize=16, fontweight='bold')
    plt.savefig(out_dir / f'Figure5_Native_vs_IVT_hexbin{suffix}.png', dpi=600, bbox_inches='tight')
    plt.savefig(out_dir / f'Figure5_Native_vs_IVT_hexbin{suffix}.pdf', bbox_inches='tight')
    print(f"Hexbin ({label}): {out_dir}/Figure5_Native_vs_IVT_hexbin{suffix}.png/pdf")
    plt.close()

    # scatter
    fig = plt.figure(figsize=(18, 6))
    gs = GridSpec(1, 3, figure=fig, hspace=0.3, wspace=0.40)
    for idx, (merged_df, stats_dict, name1, name2) in enumerate(data_list):
        ax = fig.add_subplot(gs[0, idx])
        title = f'{name1.replace(" Rep ", " Rep")} vs {name2.replace(" Rep ", " Rep")}'
        plot_scatter_panel(ax, merged_df, name1, name2, stats_dict, title, panel_labels[idx])
    fig.suptitle('', fontsize=18, fontweight='bold')
    plt.savefig(out_dir / f'Figure5_Native_vs_IVT_scatter{suffix}.png', dpi=600, bbox_inches='tight')
    plt.savefig(out_dir / f'Figure5_Native_vs_IVT_scatter{suffix}.pdf', bbox_inches='tight')
    print(f"Scatter ({label}): {out_dir}/Figure5_Native_vs_IVT_scatter{suffix}.png/pdf")
    plt.close()

Hexbin (Per-base): per_base/Figure5_Native_vs_IVT_hexbin.png/pdf
Scatter (Per-base): per_base/Figure5_Native_vs_IVT_scatter.png/pdf
Hexbin (100bp bins): binned_100bp/Figure5_Native_vs_IVT_hexbin_100bp.png/pdf
Scatter (100bp bins): binned_100bp/Figure5_Native_vs_IVT_scatter_100bp.png/pdf
Hexbin (1kb bins): binned_1kb/Figure5_Native_vs_IVT_hexbin_1kb.png/pdf
Scatter (1kb bins): binned_1kb/Figure5_Native_vs_IVT_scatter_1kb.png/pdf


## Summary table

In [79]:
all_stats = []
for resolution, config in datasets.items():
    for stats_dict in config['stats_list']:
        stats_copy = stats_dict.copy()
        stats_copy['resolution'] = config['label']
        all_stats.append(stats_copy)

summary_df = pd.DataFrame(all_stats)
summary_df['spearman_r'] = summary_df['spearman_r'].round(3)
summary_df['pearson_r'] = summary_df['pearson_r'].round(3)
summary_df = summary_df[['resolution', 'comparison', 'n_positions', 'spearman_r', 'pearson_r',
                          'n_total', 'pct_covered_both', 'spearman_p', 'pearson_p']]

summary_df.to_csv('Table_Native_vs_IVT_Correlations_all_resolutions.csv', index=False)
print(summary_df[['resolution', 'comparison', 'n_positions', 'spearman_r', 'pearson_r']].to_string(index=False))

print("\nAverages by resolution:")
for resolution in ['Per-base', '100bp bins', '1kb bins']:
    res_df = summary_df[summary_df['resolution'] == resolution]
    print(f"  {resolution}: Spearman \u03c1 = {res_df['spearman_r'].mean():.3f}, "
          f"Pearson r = {res_df['pearson_r'].mean():.3f}")

# interpretation based on 1kb binned (most statistically appropriate given per-base autocorrelation)
res_1kb = summary_df[summary_df['resolution'] == '1kb bins']
avg_spearman_1kb = res_1kb['spearman_r'].mean()
if avg_spearman_1kb >= 0.80:
    print("\nIVT RNA faithfully reproduces native RNA coverage patterns (strong correlation).")
elif avg_spearman_1kb >= 0.70:
    print("\nIVT RNA generally reproduces native RNA coverage patterns.")
else:
    print("\nModerate correlation — consider additional validation.")

resolution                comparison  n_positions  spearman_r  pearson_r
  Per-base Native Rep 1 vs IVT Rep 1       484461       0.631      0.878
  Per-base Native Rep 2 vs IVT Rep 2       506696       0.638      0.884
  Per-base Native Rep 3 vs IVT Rep 3       542924       0.629      0.892
100bp bins Native Rep 1 vs IVT Rep 1         6115       0.590      0.883
100bp bins Native Rep 2 vs IVT Rep 2         6385       0.560      0.887
100bp bins Native Rep 3 vs IVT Rep 3         6842       0.534      0.895
  1kb bins Native Rep 1 vs IVT Rep 1         1406       0.616      0.927
  1kb bins Native Rep 2 vs IVT Rep 2         1448       0.574      0.922
  1kb bins Native Rep 3 vs IVT Rep 3         1524       0.613      0.930

Averages by resolution:
  Per-base: Spearman ρ = 0.633, Pearson r = 0.885
  100bp bins: Spearman ρ = 0.561, Pearson r = 0.888
  1kb bins: Spearman ρ = 0.601, Pearson r = 0.926

Moderate correlation — consider additional validation.
